In [2]:
import sys
import pickle
import numpy as np
import re
from collections import defaultdict
sys.path.insert(0, '/Users/jl/CMU-MultimodalSDK')
sys.path.insert(0, '/Users/jl/MISA/src')
from mmsdk import mmdatasdk as md
from create_dataset import word2id, return_unk

DATA_PATH = '/Users/jl/MISA/datasets/MOSEI'

DATASET = md.cmu_mosei
train_split = set(DATASET.standard_folds.standard_train_fold)
dev_split   = set(DATASET.standard_folds.standard_valid_fold)
test_split  = set(DATASET.standard_folds.standard_test_fold)

labels   = md.computational_sequence(DATA_PATH + '/CMU_MOSEI_LabelsSentiment.csd.bak', DATA_PATH)
words_cs = md.computational_sequence(DATA_PATH + '/CMU_MOSEI_TimestampedWords.csd.bak', DATA_PATH)
visual   = md.computational_sequence(DATA_PATH + '/CMU_MOSEI_VisualFacet42.csd.bak', DATA_PATH)
acoustic = md.computational_sequence(DATA_PATH + '/CMU_MOSEI_COVAREP.csd.bak', DATA_PATH)

EPS = 1e-6
train, dev, test = [], [], []
MAX_TRAIN, MAX_DEV, MAX_TEST = 2000, 300, 500

print("Building dataset...")
for vid in list(labels.data.keys()):
    vid_id = vid.split('[')[0] if '[' in vid else vid
    if vid_id not in train_split and vid_id not in dev_split and vid_id not in test_split:
        continue
    try:
        label_feat = labels.data[vid]['features']
        word_feat  = words_cs.data[vid]['features']
        vis_feat   = visual.data[vid]['features']
        acou_feat  = acoustic.data[vid]['features']
    except:
        continue
    if len(word_feat) == 0 or len(vis_feat) == 0 or len(acou_feat) == 0:
        continue
    min_len = min(len(word_feat), len(vis_feat), len(acou_feat))
    word_feat = word_feat[:min_len]
    vis_feat  = vis_feat[:min_len]
    acou_feat = acou_feat[:min_len]
    actual_words, word_ids, vis_list, acou_list = [], [], [], []
    for i, word in enumerate(word_feat):
        try:
            w = word[0].decode('utf-8') if isinstance(word[0], bytes) else str(word[0])
        except:
            continue
        if w != 'sp':
            actual_words.append(w)
            word_ids.append(word2id[w])
            vis_list.append(vis_feat[i])
            acou_list.append(acou_feat[i])
    if len(word_ids) == 0:
        continue
    words_arr = np.asarray(word_ids)
    vis_arr   = np.nan_to_num(np.asarray(vis_list))
    acou_arr  = np.nan_to_num(np.asarray(acou_list))
    # Average across annotators, keep single sentiment score
    label_arr = np.array([[np.nanmean(label_feat[:, 0])]])
    vis_arr  = np.nan_to_num((vis_arr  - vis_arr.mean(0,  keepdims=True)) / (EPS + vis_arr.std(0,  keepdims=True)))
    acou_arr = np.nan_to_num((acou_arr - acou_arr.mean(0, keepdims=True)) / (EPS + acou_arr.std(0, keepdims=True)))
    entry = ((words_arr, vis_arr, acou_arr, actual_words), label_arr, vid)
    if vid_id in train_split and len(train) < MAX_TRAIN:
        train.append(entry)
    elif vid_id in dev_split and len(dev) < MAX_DEV:
        dev.append(entry)
    elif vid_id in test_split and len(test) < MAX_TEST:
        test.append(entry)
    if len(train) >= MAX_TRAIN and len(dev) >= MAX_DEV and len(test) >= MAX_TEST:
        break

print(f"Train: {len(train)}, Dev: {len(dev)}, Test: {len(test)}")

for split, data in [('train', train), ('dev', dev), ('test', test)]:
    with open(f'{DATA_PATH}/{split}.pkl', 'wb') as f:
        pickle.dump(data, f)
print("Base pickles saved.")

[2026-05-06 06:14:07.972] | Success | Computational sequence read from file /Users/jl/MISA/datasets/MOSEI/CMU_MOSEI_LabelsSentiment.csd.bak ...
[2026-05-06 06:14:08.169] | Status  | Checking the integrity of the <All Labels> computational sequence ...
[2026-05-06 06:14:08.169] | Status  | Checking the format of the data in <All Labels> computational sequence ...


[2026-05-06 06:14:08.503] | Success | <All Labels> computational sequence data in correct format.
[2026-05-06 06:14:08.503] | Status  | Checking the format of the metadata in <All Labels> computational sequence ...
[2026-05-06 06:14:08.503] | Warning | <All Labels> computational sequence does not have all the required metadata ... continuing 
[2026-05-06 06:14:08.505] | Success | Computational sequence read from file /Users/jl/MISA/datasets/MOSEI/CMU_MOSEI_TimestampedWords.csd.bak ...
[2026-05-06 06:14:09.071] | Status  | Checking the integrity of the <words> computational sequence ...
[2026-05-06 06:14:09.071] | Status  | Checking the format of the data in <words> computational sequence ...


[2026-05-06 06:14:09.447] | Success | <words> computational sequence data in correct format.
[2026-05-06 06:14:09.447] | Status  | Checking the format of the metadata in <words> computational sequence ...
[2026-05-06 06:14:09.447] | Warning | <words> computational sequence does not have all the required metadata ... continuing 
[2026-05-06 06:14:09.448] | Success | Computational sequence read from file /Users/jl/MISA/datasets/MOSEI/CMU_MOSEI_VisualFacet42.csd.bak ...
[2026-05-06 06:14:09.817] | Status  | Checking the integrity of the <OpenFace_2> computational sequence ...
[2026-05-06 06:14:09.817] | Status  | Checking the format of the data in <OpenFace_2> computational sequence ...


[2026-05-06 06:14:10.667] | Success | <OpenFace_2> computational sequence data in correct format.
[2026-05-06 06:14:10.667] | Status  | Checking the format of the metadata in <OpenFace_2> computational sequence ...
[2026-05-06 06:14:10.667] | Warning | <OpenFace_2> computational sequence does not have all the required metadata ... continuing 
[2026-05-06 06:14:10.668] | Success | Computational sequence read from file /Users/jl/MISA/datasets/MOSEI/CMU_MOSEI_COVAREP.csd.bak ...
[2026-05-06 06:14:10.989] | Status  | Checking the integrity of the <COVAREP> computational sequence ...
[2026-05-06 06:14:10.989] | Status  | Checking the format of the data in <COVAREP> computational sequence ...


/Users/jl/micromamba/envs/multimodal/lib/python3.11/site-packages/numpy/core/_methods.py:118: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/Users/jl/micromamba/envs/multimodal/lib/python3.11/site-packages/numpy/core/_methods.py:152: RuntimeWarning: overflow encountered in reduce
  arrmean = umr_sum(arr, axis, dtype, keepdims=True, where=where)
/var/folders/x5/v5smrj3d2mv36k9vry639c9c0000gn/T/ipykernel_18275/4110515693.py:64: RuntimeWarning: invalid value encountered in divide
  acou_arr = np.nan_to_num((acou_arr - acou_arr.mean(0, keepdims=True)) / (EPS + acou_arr.std(0, keepdims=True)))
/Users/jl/micromamba/envs/multimodal/lib/python3.11/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


[2026-05-06 06:14:11.832] | Success | <COVAREP> computational sequence data in correct format.
[2026-05-06 06:14:11.832] | Status  | Checking the format of the metadata in <COVAREP> computational sequence ...
[2026-05-06 06:14:11.832] | Warning | <COVAREP> computational sequence does not have all the required metadata ... continuing 
Building dataset...
Train: 2000, Dev: 300, Test: 500
Base pickles saved.


In [3]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from tqdm import tqdm

DATA_PATH = '/Users/jl/MISA/datasets/MOSEI'

print("Loading DistilBERT...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')
bert_model.eval()

def extract_bert_features(actual_words, max_len=64):
    text = ' '.join(actual_words)
    tokens = tokenizer(text, max_length=max_len, padding='max_length',
                       truncation=True, return_tensors='pt')
    with torch.no_grad():
        output = bert_model(**tokens)
    return output.last_hidden_state[:, 0, :].squeeze().numpy()

for split in ['train', 'dev', 'test']:
    print(f"\nProcessing {split}...")
    with open(f'{DATA_PATH}/{split}.pkl', 'rb') as f:
        data = pickle.load(f)
    enriched = []
    for (words, vis, acou, actual_words), label, vid in tqdm(data):
        bert_feat = extract_bert_features(actual_words)
        enriched.append(((words, vis, acou, actual_words, bert_feat), label, vid))
    with open(f'{DATA_PATH}/{split}_bert.pkl', 'wb') as f:
        pickle.dump(enriched, f)
    print(f"Saved {split}_bert.pkl — {len(enriched)} samples")

del bert_model
print("\nDone!")

Loading DistilBERT...


Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertModel: ['vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_projector.bias', 'vocab_transform.bias', 'vocab_layer_norm.weight']
- This IS expected if you are initializing DistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



Processing train...


100%|██████████| 2000/2000 [00:38<00:00, 51.83it/s]


Saved train_bert.pkl — 2000 samples

Processing dev...


100%|██████████| 300/300 [00:05<00:00, 51.84it/s]


Saved dev_bert.pkl — 300 samples

Processing test...


100%|██████████| 500/500 [00:10<00:00, 46.15it/s]


Saved test_bert.pkl — 500 samples

Done!
